# مشروع Vmem المتنبَّأ به — الورم الدبقي (GBM)

**السؤال:** هل الجهد الكهربائي المتوقع لخلايا الورم (محسوباً من الجينات النشطة بمعادلة GHK) يتنبأ ببقاء المريض؟

**طريقة التشغيل:** من القائمة العلوية اختر **Runtime ← Run all** (أو Ctrl+F9) وانتظر دقيقة تقريباً.
لا تعدّل شيئاً. النتيجة النهائية تظهر في آخر خليتين.

In [ ]:
!pip install lifelines requests pandas matplotlib -q
print("تم التثبيت")

## 1) المحرك الفيزيائي (معادلة GHK) — بلا ذكاء اصطناعي

In [ ]:
import numpy as np

R, F, T_BODY = 8.314, 96485, 310.15
CONC = dict(K_out=5.0, K_in=140.0, Na_out=145.0, Na_in=15.0, Cl_in=10.0, Cl_out=110.0)
BASELINE = dict(p_k=1.0, p_na=0.02, p_cl=0.10)

def rtf_mV(T=T_BODY):
    return (R * T) / F * 1000.0

def ghk_vmem(p_k, p_na, p_cl):
    num = p_k * CONC["K_out"] + p_na * CONC["Na_out"] + p_cl * CONC["Cl_in"]
    den = p_k * CONC["K_in"] + p_na * CONC["Na_in"] + p_cl * CONC["Cl_out"]
    return rtf_mV() * np.log(num / den)

def expr_to_scale(value, reference, max_scale=5.0):
    """تحويل log2 (لأن بيانات RNA-seq منحرفة بشدة) مع سقف أمان."""
    if reference is None or reference <= 0 or np.isnan(value):
        return 1.0
    scale = np.log2(max(value, 1e-6) + 1) / np.log2(reference + 1)
    return float(np.clip(scale, 1.0 / max_scale, max_scale))

def sample_vmem(gene_expr, reference, gene_map):
    scales = {"K": [], "Na": [], "Cl": []}
    for gene, ch in gene_map.items():
        if gene in gene_expr and gene in reference:
            scales[ch].append(expr_to_scale(gene_expr[gene], reference[gene]))
    agg = lambda ch: float(np.mean(scales[ch])) if scales[ch] else 1.0
    return ghk_vmem(BASELINE["p_k"] * agg("K"),
                    BASELINE["p_na"] * agg("Na"),
                    BASELINE["p_cl"] * agg("Cl"))

# اختبار ذاتي: يجب أن يطابق الحساب اليدوي (-75.7 و -39.6)
print("خط الأساس:", round(ghk_vmem(1.0, 0.02, 0.10), 2), "mV  (المتوقع ≈ -75.7)")
print("بعد زيادة Na ×10:", round(ghk_vmem(1.0, 0.20, 0.10), 2), "mV  (المتوقع ≈ -39.6)")

## 2) جلب البيانات الحقيقية من cBioPortal (TCGA)

In [ ]:
import requests
import pandas as pd

API = "https://www.cbioportal.org/api"
STUDY = "gbm_tcga_pan_can_atlas_2018"

# جين -> نوع القناة (GJ لا تدخل معادلة GHK مباشرة)
CHANNEL_GENE_MAP = {
    "SCN9A": "Na",
    "KCNJ10": "K", "KCNJ2": "K",
    "CLIC1": "Cl",
    "GJB2": "GJ", "GJA1": "GJ",
}

def entrez_map(symbols):
    r = requests.post(f"{API}/genes/fetch", params={"geneIdType": "HUGO_GENE_SYMBOL"},
                      json=symbols, timeout=30)
    r.raise_for_status()
    return {g["hugoGeneSymbol"]: g["entrezGeneId"] for g in r.json()}

def fetch_expression(study, symbols):
    em = entrez_map(symbols)
    r = requests.post(
        f"{API}/molecular-profiles/{study}_rna_seq_v2_mrna/molecular-data/fetch",
        params={"projection": "SUMMARY"},
        json={"entrezGeneIds": list(em.values()), "sampleListId": f"{study}_rna_seq_v2_mrna"},
        timeout=60)
    r.raise_for_status()
    inv = {v: k for k, v in em.items()}
    rows = {}
    for rec in r.json():
        rows.setdefault(rec["sampleId"], {})[inv.get(rec["entrezGeneId"])] = rec.get("value")
    return pd.DataFrame.from_dict(rows, orient="index").apply(pd.to_numeric, errors="coerce")

def fetch_survival(study):
    r = requests.get(f"{API}/studies/{study}/clinical-data",
                     params={"clinicalDataType": "PATIENT", "projection": "SUMMARY"}, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df = df[df["clinicalAttributeId"].isin(["OS_STATUS", "OS_MONTHS"])]
    return df.pivot_table(index="patientId", columns="clinicalAttributeId",
                          values="value", aggfunc="first")

expr_df = fetch_expression(STUDY, list(CHANNEL_GENE_MAP))
surv_df = fetch_survival(STUDY)
print(f"التعبير الجيني: {expr_df.shape[0]} عيّنة × {expr_df.shape[1]} جينات")
print(f"بيانات البقاء: {surv_df.shape[0]} مريض")
expr_df.head()

## 3) حساب Vmem المتنبَّأ به لكل مريض

In [ ]:
ghk_map = {g: t for g, t in CHANNEL_GENE_MAP.items() if t in ("K", "Na", "Cl")}
reference = expr_df.mean(axis=0).to_dict()   # المرجع: متوسط عيّنات الدراسة نفسها (تقريب أولي)

vmem = pd.Series(
    {sid: sample_vmem(row.to_dict(), reference, ghk_map) for sid, row in expr_df.iterrows()},
    name="predicted_vmem_mV")

print("توزيع Vmem المتنبَّأ به:")
print(vmem.describe().round(2))
vmem.to_csv("gbm_predicted_vmem.csv")

## 4) هل Vmem يتنبأ بالبقاء؟ (النتيجة العلمية الأولى)

In [ ]:
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt

v = vmem.copy()
v.index = [i[:12] for i in v.index]          # باركود المريض = أول 12 حرفاً
v = v[~v.index.duplicated(keep="first")]

d = surv_df.join(v, how="inner").copy()
d["OS_MONTHS"] = pd.to_numeric(d["OS_MONTHS"], errors="coerce")
d["event"] = d["OS_STATUS"].astype(str).str.startswith("1").astype(int)
d = d.dropna(subset=["OS_MONTHS", "predicted_vmem_mV"])
print(f"عدد المرضى في التحليل: {len(d)}  |  وفيات: {d['event'].sum()}")

cph = CoxPHFitter()
cph.fit(d[["OS_MONTHS", "event", "predicted_vmem_mV"]], duration_col="OS_MONTHS", event_col="event")
res = cph.summary.loc["predicted_vmem_mV"]
print()
print(cph.summary[["coef", "exp(coef)", "p"]].round(4))

print("\n===== الخلاصة بكلمات بسيطة =====")
if res["p"] < 0.05:
    direction = "أعلى (أقرب للاستقطاب)" if res["coef"] > 0 else "أقل"
    print(f"✅ توجد علاقة ذات دلالة إحصائية (p = {res['p']:.4f}).")
    print(f"   كلما كان Vmem المتوقع {direction}، ارتبط ذلك بخطر وفاة أعلى" if res["coef"] > 0
          else "   كلما ارتفع Vmem المتوقع، قلّ خطر الوفاة")
else:
    print(f"❌ لا توجد علاقة ذات دلالة إحصائية (p = {res['p']:.4f}).")
    print("   هذه نتيجة حقيقية وتُوثَّق بصدق؛ لا تعني فشل المشروع.")
    print("   (الأسباب المحتملة: عدد قليل من الجينات، عيّنة صغيرة، عوامل مثل العمر لم تُضبط)")

In [ ]:
med = d["predicted_vmem_mV"].median()
hi = d["predicted_vmem_mV"] >= med
km = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(7, 5))
km.fit(d.loc[hi, "OS_MONTHS"], d.loc[hi, "event"], label=f"Vmem أعلى من الوسيط (n={hi.sum()})")
km.plot_survival_function(ax=ax)
km.fit(d.loc[~hi, "OS_MONTHS"], d.loc[~hi, "event"], label=f"Vmem أقل من الوسيط (n={(~hi).sum()})")
km.plot_survival_function(ax=ax)
lr = logrank_test(d.loc[hi, "OS_MONTHS"], d.loc[~hi, "OS_MONTHS"],
                  d.loc[hi, "event"], d.loc[~hi, "event"])
ax.set_title(f"Kaplan-Meier — log-rank p = {lr.p_value:.4f}")
ax.set_xlabel("Months"); ax.set_ylabel("Survival probability")
plt.tight_layout(); plt.savefig("km_vmem.png", dpi=150); plt.show()

## كيف تقرأ النتيجة

- **p < 0.05** ⟵ علاقة حقيقية إحصائياً بين Vmem المتوقع والبقاء.
- **coef موجب** ⟵ Vmem أعلى (أقرب للصفر = أكثر استقطاباً) يرتبط بخطر أعلى، وهذا يوافق فرضية ليفين.
- **p ≥ 0.05** ⟵ لا علاقة واضحة. نتيجة صادقة ولا تعني فشل المشروع.

**حدود يجب ذكرها دائماً:** mRNA ليس نشاط القناة الفعلي، والمرجع هو متوسط الأورام نفسها وليس نسيجاً سليماً، والجينات 6 فقط.